# Colab GPU training — Real-Time Threat Detection

Reproduces paper Table II (YOLOv8, 50 & 100 epochs) on a Colab GPU.

**Before running:** Runtime → Change runtime type → **GPU**.

Set `REPO_URL` to your GitHub clone URL after pushing.

In [ ]:
# <<< EDIT THIS after you push to GitHub >>>
REPO_URL = "https://github.com/jon44ai-svg/realtime-threat-detection.git"
BRANCH = "feat/initial-workspace"

import os, pathlib
ROOT = pathlib.Path("/content/realtime-threat-detection")
if not ROOT.exists():
    !git clone --branch {BRANCH} {REPO_URL} {ROOT}
%cd {ROOT}
!pip install -q ultralytics opencv-python-headless pyyaml matplotlib seaborn kaggle python-dotenv pillow
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## Kaggle credentials
Upload `kaggle.json` via the Files sidebar, or paste username/key below.

In [ ]:
from pathlib import Path
import json, os

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
uploaded = Path("/content/kaggle.json")
if uploaded.exists():
    dest = kaggle_dir / "kaggle.json"
    dest.write_text(uploaded.read_text())
    os.chmod(dest, 0o600)
    print("Using /content/kaggle.json")
else:
    # Or set Colab Secrets / fill these:
    USERNAME = ""  # optional if key file has username
    KEY = ""
    if not KEY:
        raise SystemExit("Upload /content/kaggle.json or set KEY in this cell")
    (kaggle_dir / "kaggle.json").write_text(json.dumps({"username": USERNAME, "key": KEY}))
    os.chmod(kaggle_dir / "kaggle.json", 0o600)
    print("Wrote ~/.kaggle/kaggle.json from cell vars")

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "src"))
from threat_detection.data.download import download_dataset, write_data_yaml

ds = download_dataset(ROOT / "data" / "raw")
write_data_yaml(ds, ROOT / "configs" / "data.yaml")
print("dataset:", ds)

## Train 50 then 100 epochs (paper Table II)

In [ ]:
!python scripts/train.py --epochs 50 --device 0
!python scripts/evaluate.py --weights runs/detect/train_50/weights/best.pt --epochs 50

In [ ]:
!python scripts/train.py --epochs 100 --device 0
!python scripts/evaluate.py --weights runs/detect/train_100/weights/best.pt --epochs 100

Download `runs/detect/train_100/weights/best.pt` and `runs/val/` reports from the Colab file browser when finished.